# XRD Rietveld Plot Generator

Publication-quality Rietveld plots from the **CSV that the GSAS-II Rietveld
plot saves** - batch processing, built-in validation, cross-platform.

Not the file from *Export → Powder data as → histogram CSV file*: that one
has a quoted preamble and different column names, and is rejected.

Full documentation (input format, usage, configuration, privacy notes):
see [`README.md`](README.md).

## 1. Setup and core routines

Dependency check, then the parsing, data-preparation and plotting
functions. Plot appearance (2θ window, colours, line widths, fonts) is
controlled by the constants at the top of the second cell. Input format
and numerical-precision details are documented in the README.

In [ ]:
# Dependency bootstrap - installs only if missing (useful on Google Colab).
import importlib.util, subprocess, sys

for pkg in ("numpy", "pandas", "matplotlib", "ipywidgets"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
print("Dependencies OK")

In [ ]:
"""Core routines: GSAS-II CSV parsing, data preparation, Rietveld plot."""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import is_color_like
from matplotlib.lines import Line2D

# --- Plot appearance --------------------------------------------------
# 2theta window (deg). None takes the measured range of each file, so a
# pattern is never cropped without being asked. Set both to fix every
# figure to the same window, or give x_min and x_max per sample in the
# metadata file, which wins over these.
PLOT_X_MIN, PLOT_X_MAX = None, None
FIGURE_WIDTH, FIGURE_HEIGHT = 12, 10   # inches
DPI_EXPORT = 600                       # PNG export resolution

# Lower panel: True draws diff/sigma, the residual divided by the standard
# deviation of the point, so a well-fitted pattern stays inside a band of a
# few units. False draws the raw diff in counts, where the tall reflections
# dominate. Files carry the suffix '_counts' when this is False.
WEIGHTED_RESIDUALS = True

COLOR_OBS = "#000000"
COLOR_CALC = "#D62728"
COLOR_BKG = "#7f7f7f"
COLOR_RESIDUALS = "#696969"

LINEWIDTH_BORDER = 2.0
LINEWIDTH_CALC = 1.5
LINEWIDTH_RESIDUALS = 0.8
LINEWIDTH_TICKS = 2.0
MARKER_SIZE_OBS = 15

FONT_SIZE_LABEL = 20
FONT_SIZE_LEGEND = 16
FONT_SIZE_TICK = 16

# --- Phase columns ----------------------------------------------------
# Headers GSAS-II writes itself. Everything else that holds only a few
# values is taken to be the reflection positions of a phase, so phases are
# found by their own names instead of a keyword list.
NON_PHASE_COLUMNS = frozenset({
    "used", "obs", "calc", "bkg", "diff", "diff/sigma", "weight", "weights",
    "sig", "sigma", "tick-pos", "tick pos", "axis-limits", "axis limits",
    "excluded", "gof",
})
# A reflection list is short: at most this share of the pattern's rows. A
# column above the ceiling is reported and left undrawn, never dropped in
# silence.
PHASE_MAX_FILL = 0.5
# Legend name for a phase whose column header is not the name to print.
# Keys are matched case-insensitively as a fragment of the header, and the
# longest matching key wins:
#   PHASE_LABELS = {"phase 2": "Phase 2, high temperature"}
PHASE_LABELS = {}
# Tick colour tied to a phase name rather than to a row, so a phase keeps
# its colour whether or not the others are present in that sample. Keys are
# matched as a fragment of the legend name, the name PHASE_LABELS produced:
#   PHASE_COLORS = {"phase 1": "#1f77b4"}
# The same is available per sample, and privately, through the
# '<phase>_color' columns of the metadata file, which win over this.
PHASE_COLORS = {}
# Colours for every phase with no colour of its own, in legend order.
PHASE_COLOR_CYCLE = ("#EE8031", "#1f77b4", "#FF1493", "#2CA02C", "#9467BD",
                     "#8C564B")


def fmt_pct(p):
    """Phase fraction label: one decimal, trailing '.0' dropped (62.0 -> 62)."""
    s = f"{p:.1f}"
    return s[:-2] if s.endswith(".0") else s


def residual_column(weighted=None):
    """Header the lower panel is drawn from, per WEIGHTED_RESIDUALS."""
    if weighted is None:
        weighted = WEIGHTED_RESIDUALS
    return "diff/sigma" if weighted else "diff"


def plot_window(theta, x_min=None, x_max=None):
    """2theta limits: the per-sample pair, then the constants, then the data."""
    low = x_min if x_min is not None else PLOT_X_MIN
    high = x_max if x_max is not None else PLOT_X_MAX
    return (float(np.nanmin(theta)) if low is None else float(low),
            float(np.nanmax(theta)) if high is None else float(high))


def phase_label(column):
    """Legend name for a phase column: 'Phase 1 hkl' -> 'Phase 1'."""
    label = column.strip()
    if label.lower().endswith("hkl"):
        label = label[:-3].strip()
    override = longest_match(PHASE_LABELS, label)
    return override if override else label[:1].upper() + label[1:]


def longest_match(mapping, label):
    """Value whose key is the longest fragment of label, or None.

    The longest key wins so that a specific column ('phase 1_color') beats a
    general one ('phase_color') instead of resolving on dictionary order.
    """
    keys = [k for k in (mapping or {}) if k and k in label.lower()]
    return mapping[max(keys, key=len)] if keys else None


def phase_colors(ordered_labels, overrides=None):
    """Tick colour per phase: the metadata, then PHASE_COLORS, then the cycle.

    Both mappings are keyed by a fragment of the legend name. Repeated
    labels share one colour and consume one slot of the cycle, matching the
    single legend entry they get.
    """
    colors, cycled = {}, 0
    for label in dict.fromkeys(ordered_labels):
        color = longest_match(overrides, label) or longest_match(PHASE_COLORS,
                                                                 label)
        if color is None:
            color = PHASE_COLOR_CYCLE[cycled % len(PHASE_COLOR_CYCLE)]
            cycled += 1
        colors[label] = color
    return colors


def phase_fraction(pct, label):
    """Percentage for one phase, matched on the metadata key as a fragment."""
    hits = sorted(k for k in pct if k in label.lower())
    if len(hits) > 1:
        print(f"Warning: metadata columns {', '.join(h + '_pct' for h in hits)}"
              f" all match '{label}' - no percentage printed.")
        return 0.0
    return pct[hits[0]] if hits else 0.0


# --- Parsing -----------------------------------------------------------
def read_gsas2_csv(csv_path, weighted=None):
    """Read one CSV saved from a GSAS-II Rietveld plot.

    Returns (data, phase_cols, error): on success error is None, on failure
    data is None and error holds the reason. Non-fatal issues are printed
    as warnings. 'weighted' overrides WEIGHTED_RESIDUALS for this call.
    """
    csv_path = Path(csv_path)
    warnings = []
    try:
        # Columns are read as strings: pandas' fast C float parser is not
        # correctly rounded (up to 1 ULP off), Python's float() is.
        # GSAS-II uses ';' or ',' as separator depending on locale.
        try:
            df = pd.read_csv(csv_path, sep=";", encoding="utf-8", dtype=str)
            if len(df.columns) <= 1:
                df = pd.read_csv(csv_path, sep=",", encoding="utf-8", dtype=str)
        except (pd.errors.ParserError, UnicodeDecodeError):
            # Ragged or mis-encoded file. The python engine hands each bad
            # row to a callback rather than giving up on the whole file,
            # and dtype=str keeps the surviving rows correctly rounded.
            bad = []
            for sep in (";", ","):
                bad.clear()
                df = pd.read_csv(csv_path, sep=sep, encoding="latin-1",
                                 dtype=str, engine="python",
                                 on_bad_lines=lambda row: bad.append(row))
                if len(df.columns) > 1:
                    break
            if bad:
                warnings.append(f"{len(bad)} malformed row(s) skipped")

        df.columns = df.columns.str.strip()
        if df.empty:
            return None, None, "empty CSV file"

        col_lower_map = {c.lower(): c for c in df.columns}

        def clean(s):
            """Correctly-rounded str -> float64, ',' decimal mark accepted."""
            def to_float(v):
                try:
                    return float(v.replace(",", "."))
                except (AttributeError, ValueError, TypeError):
                    return np.nan  # missing or non-numeric entry
            return np.array([to_float(v) for v in s], dtype=np.float64)

        data = {}

        # 2theta column: header contains '2theta' or starts like 'x,'.
        theta = next((col for cl, col in col_lower_map.items()
                      if "2theta" in cl or "x," in cl), None)
        if theta is None:
            return None, None, "2theta column not found"
        data["x"] = clean(df[theta])
        if np.all(np.isnan(data["x"])):
            return None, None, "no valid data in 2theta column"

        # Main pattern columns, in the fixed GSAS-II layout.
        missing = []
        for k in ("obs", "calc", "bkg"):
            col = col_lower_map.get(k)
            if col:
                data[k] = clean(df[col])
                n_nan = int(np.isnan(data[k]).sum())
                if n_nan:
                    warnings.append(f"column '{k}': {n_nan} non-numeric values -> NaN")
            else:
                data[k] = np.zeros_like(data["x"])
                missing.append(k)
        if missing:
            warnings.append(f"missing columns (using zeros): {', '.join(missing)}")
        if np.all(np.isnan(data["obs"])):
            return None, None, "no valid data in the obs column"

        # Residuals are not filled with zeros when absent: a flat lower
        # panel reads as a perfect fit, which is the one lie the figure
        # must not tell.
        wanted = residual_column(weighted)
        resid_col = col_lower_map.get(wanted)
        if resid_col is None:
            return None, None, f"residual column '{wanted}' not found"
        data["resid"] = clean(df[resid_col])
        n_nan = int(np.isnan(data["resid"]).sum())
        if n_nan:
            warnings.append(f"column '{wanted}': {n_nan} non-numeric values -> NaN")

        # Per-phase reflection-position columns: whatever is left over and
        # sparse enough to be a reflection list rather than a data column.
        phase_cols = []
        for cl, col in col_lower_map.items():
            # pandas renames a repeated header 'tick-pos' to 'tick-pos.1',
            # which the blocklist would otherwise miss.
            head, _, tail = cl.rpartition(".")
            base = head if head and tail.isdigit() else cl
            if col == theta or base in NON_PHASE_COLUMNS:
                continue
            vals = clean(df[col])
            n_values = int(np.count_nonzero(~np.isnan(vals)))
            if not n_values:
                continue
            if n_values > PHASE_MAX_FILL * len(vals):
                warnings.append(f"column '{col}': {n_values} values in "
                                f"{len(vals)} rows, too many for a reflection "
                                "list, not drawn")
                continue
            phase_cols.append(col)
            data[col] = vals

        if warnings:
            print(f"[{csv_path.name}] " + "; ".join(warnings))
        print(f"[{csv_path.name}] phases detected: "
              + (", ".join(sorted(phase_cols, key=phase_label)) or "none"))
        return data, phase_cols, None

    except Exception as e:  # isolate unreadable files, keep the batch running
        return None, None, f"read error: {e}"


def load_metadata(metadata_path):
    """Read the (private) sample metadata CSV, indexed by filename.

    Expected columns: 'filename' (or 'file'), optional 'formula', one
    '<phase>_pct' and one '<phase>_color' column per phase, and an
    optional 'x_min'/'x_max' pair. Returns an empty DataFrame when the
    file is absent. The contents are never displayed by this notebook.
    """
    metadata_path = Path(metadata_path)
    if not metadata_path.is_file():
        return pd.DataFrame()
    try:
        df = pd.read_csv(metadata_path, sep=";", encoding="utf-8", dtype=str)
        if len(df.columns) <= 1:
            df = pd.read_csv(metadata_path, sep=",", encoding="utf-8", dtype=str)
    except (pd.errors.ParserError, UnicodeDecodeError):
        df = pd.read_csv(metadata_path, sep=",", encoding="latin-1", dtype=str)

    df.columns = df.columns.str.strip().str.lower()
    fname_col = next((c for c in df.columns if c in ("filename", "file")), None)
    if fname_col is None:
        print("Warning: metadata needs a 'filename' or 'file' column - ignored.")
        return pd.DataFrame()
    df["filename"] = df[fname_col].astype(str).str.strip()
    df = df.drop_duplicates("filename", keep="first").set_index("filename")
    print(f"Metadata loaded for {len(df)} sample(s).")
    return df


def to_number(value):
    """One metadata cell as a float, or None when it is blank or not a number."""
    try:
        number = float(str(value).replace(",", "."))  # decimal comma
    except (ValueError, TypeError):
        return None
    return None if np.isnan(number) else number


def to_color(value):
    """One metadata cell as a colour, or None when blank or not a colour."""
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return None
    if not is_color_like(text):
        print(f"Warning: '{text}' is not a colour, the cycle is used instead.")
        return None
    return text


def metadata_keys(columns, suffix):
    """Column names ending in suffix, keyed by the phase fragment before it.

    A column named exactly like the suffix carries no phase name and is
    skipped, since its empty key would match every phase.
    """
    return {c[:-len(suffix)].replace("_", " ").strip(): c for c in columns
            if c.endswith(suffix) and len(c) > len(suffix)}


def sample_info(meta_df, filename, default_name):
    """Name, fractions, colours and 2theta window for one file (empty-safe).

    Every '<phase>_pct' and '<phase>_color' column becomes one entry, keyed
    by the part before the suffix with underscores as spaces, so
    'phase_1_color' reaches the phase named 'Phase 1' in the figure.
    'x_min' and 'x_max' are returned as given, for plot_window to apply.
    """
    name, pct, colors, window = default_name, {}, {}, (None, None)
    if filename not in meta_df.index:
        return name, pct, colors, window

    row = meta_df.loc[filename]
    formula = row.get("formula")
    if isinstance(formula, str) and formula.strip():
        name = formula.strip()  # an empty cell is NaN, which would print as 'nan'

    for key, col in metadata_keys(meta_df.columns, "_pct").items():
        value = to_number(row[col])
        if value is not None:
            pct[key] = value
    for key, col in metadata_keys(meta_df.columns, "_color").items():
        value = to_color(row[col])
        if value is not None:
            colors[key] = value

    window = (to_number(row.get("x_min")), to_number(row.get("x_max")))
    return name, pct, colors, window


# --- Data preparation ---------------------------------------------------
def prepare_data(data, phase_cols, use_sqrt=True):
    """Mask invalid points; optionally apply the display-only sqrt transform."""
    mask = ~(np.isnan(data["x"]) | np.isnan(data["obs"]))
    if not mask.any():
        raise ValueError("no point has both a 2theta and an obs value")
    x = data["x"][mask]

    if use_sqrt:
        obs = np.sqrt(np.abs(data["obs"][mask]))
        calc = np.sqrt(np.abs(data["calc"][mask]))
        bkg = np.sqrt(np.abs(data["bkg"][mask]))
    else:
        obs, calc, bkg = (data[k][mask] for k in ("obs", "calc", "bkg"))

    resid = data["resid"][mask]

    # Phase columns hold independent reflection positions (different length
    # from the pattern): drop their NaN padding individually.
    phases = {k: data[k][~np.isnan(data[k])] for k in phase_cols}
    return x, obs, calc, bkg, resid, phases


# --- Plotting -------------------------------------------------------------
def create_plot(theta, obs, calc, bkg, resid, phases, name, pct,
                use_sqrt=True, xlim=(None, None), ylim=(None, None),
                colors=None, weighted=None):
    """Two-panel Rietveld plot; returns the matplotlib Figure."""
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(FIGURE_WIDTH, FIGURE_HEIGHT), dpi=110,
        gridspec_kw={"height_ratios": [4, 1]}, sharex=True)

    ax1.plot(theta, bkg, color=COLOR_BKG, ls="--", lw=1.5,
             label="_nolegend_", zorder=1)
    ax1.plot(theta, calc, color=COLOR_CALC, lw=LINEWIDTH_CALC,
             label="Calculated Fit", zorder=3)
    ax1.scatter(theta, obs, color=COLOR_OBS, s=MARKER_SIZE_OBS,
                label="Observed", zorder=2, lw=0)

    # Reflection tick rows below the pattern, one row per phase, in the
    # order of the legend. The intensity axis is scaled on the part of the
    # pattern the window actually shows, so cropping in 2theta does not
    # leave the figure scaled by a peak nobody sees.
    x_low, x_high = plot_window(theta, *xlim)
    visible = (theta >= x_low) & (theta <= x_high)
    shown = obs[visible] if visible.any() else obs
    ymin, ymax = shown.min(), shown.max()
    yrange = ymax - ymin
    base_y, step_y, tick_h = (ymin - 0.05 * yrange, 0.06 * yrange,
                              0.04 * yrange)

    labels = {ph: phase_label(ph) for ph in phases}
    ordered = sorted(phases, key=lambda ph: labels[ph])
    tick_colors = phase_colors([labels[ph] for ph in ordered], colors)
    rows = 0
    for ph in ordered:
        locs = phases[ph][phases[ph] > 0.1]
        if len(locs):
            y = base_y - rows * step_y
            ax1.vlines(locs, y, y + tick_h, colors=tick_colors[labels[ph]],
                       lw=LINEWIDTH_TICKS, zorder=4)
            rows += 1

    ylabel = r"$\sqrt{Counts}$ / (a.u.)" if use_sqrt else r"Counts / (a.u.)"
    ax1.set_ylabel(ylabel, fontsize=FONT_SIZE_LABEL, labelpad=10)
    ax1.set_yticklabels([])
    ax1.tick_params(axis="y", length=0)
    ax1.tick_params(direction="in", top=False, right=False, left=True,
                    width=1.5, length=6, labelsize=FONT_SIZE_TICK)
    for spine in ax1.spines.values():
        spine.set_linewidth(LINEWIDTH_BORDER)
    ax1.spines["bottom"].set_visible(False)
    ax1.tick_params(bottom=False)
    # Default limits leave room for the tick rows below and a margin above;
    # both ends can be given explicitly, in the units actually drawn (sqrt
    # counts when USE_SQRT).
    y_low, y_high = ylim
    ax1.set_ylim(bottom=(base_y - (max(rows, 1) - 0.5) * step_y
                         if y_low is None else float(y_low)),
                 top=(ymax + 0.05 * yrange if y_high is None
                      else float(y_high)))

    # Legend: pattern entries, then one entry per detected phase.
    handles = [
        Line2D([0], [0], color=COLOR_OBS, marker="o", ls="", label=name),
        Line2D([0], [0], color=COLOR_CALC, lw=2, label="Calculated Fit"),
        Line2D([0], [0], color=COLOR_BKG, lw=2, ls="--", label="Background"),
    ]
    seen = set()
    for ph in ordered:
        lbl = labels[ph]
        if lbl in seen:
            continue  # two columns of the same phase share one entry
        seen.add(lbl)
        frac = phase_fraction(pct, lbl)
        text = f"{lbl} ({fmt_pct(frac)}%)" if frac > 0 else lbl
        handles.append(Line2D([0], [0], color=tick_colors[lbl], lw=3,
                              label=text))
    ax1.legend(handles=handles, loc="upper right", fontsize=FONT_SIZE_LEGEND,
               framealpha=1, edgecolor="black")

    orphans = sorted(k for k in pct
                     if not any(k in lbl.lower() for lbl in seen))
    if orphans:
        print("Warning: no phase in this file matches metadata column(s): "
              + ", ".join(o + "_pct" for o in orphans))

    ax2.plot(theta, resid, color=COLOR_RESIDUALS, lw=LINEWIDTH_RESIDUALS)
    ax2.axhline(0, color="black", lw=1.5)
    ax2.set_xlabel(r"2$\theta$ / ($^\circ$)", fontsize=FONT_SIZE_LABEL)
    ax2.set_ylabel(r"diff/$\sigma$" if residual_column(weighted) == "diff/sigma"
                   else r"diff / (a.u.)", fontsize=FONT_SIZE_LABEL)
    ax2.set_xlim(x_low, x_high)
    ax2.tick_params(direction="in", right=False, left=True, bottom=True,
                    width=1.5, length=6, labelsize=FONT_SIZE_TICK)
    for spine in ax2.spines.values():
        spine.set_linewidth(LINEWIDTH_BORDER)
    ax2.spines["top"].set_visible(False)
    ax2.tick_params(top=False)
    plt.subplots_adjust(hspace=0)
    return fig


def report_colors(phase_cols, colors):
    """Print the colour each detected phase was drawn with."""
    labels = sorted(phase_label(c) for c in phase_cols)
    if not labels:
        return
    drawn = phase_colors(labels, colors)
    print("  colours: " + ", ".join(f"{lbl} {drawn[lbl]}"
                                    for lbl in dict.fromkeys(labels)))


def replot_file(csv_path, metadata_file, x_min=None, x_max=None,
                y_min=None, y_max=None, use_sqrt=True, weighted=True):
    """Draw one file with the given window and toggles, without saving it.

    Returns (figure, metadata_line): the line is the row to paste into the
    metadata file so that the batch run reproduces this 2theta window.
    Raises ValueError when the file cannot be read or drawn.
    """
    csv_path = Path(csv_path)
    data, phase_cols, error = read_gsas2_csv(csv_path, weighted=weighted)
    if error:
        raise ValueError(f"{csv_path.name}: {error}")
    name, pct, colors, _ = sample_info(load_metadata(metadata_file),
                                       csv_path.name, csv_path.stem)
    report_colors(phase_cols, colors)
    fig = create_plot(*prepare_data(data, phase_cols, use_sqrt=use_sqrt),
                      name, pct, use_sqrt=use_sqrt, xlim=(x_min, x_max),
                      ylim=(y_min, y_max), colors=colors, weighted=weighted)

    cells = [csv_path.name, name,
             "" if x_min is None else f"{float(x_min):g}",
             "" if x_max is None else f"{float(x_max):g}"]
    return fig, "filename;formula;x_min;x_max\n" + ";".join(cells)


# --- Batch driver ---------------------------------------------------------
def process_folder(data_folder, metadata_file, output_folder,
                   use_sqrt=True, show=True):
    """Plot every GSAS-II CSV export in data_folder; save PDF + PNG."""
    data_dir = Path(data_folder)
    data_dir.mkdir(exist_ok=True)  # first run: created empty, ready for your files

    meta_name = Path(metadata_file).name
    files = sorted(f for f in data_dir.glob("*.csv") if f.name != meta_name)
    if not files:
        print(f"No CSV files found in '{data_dir}': add your GSAS-II exports and re-run.")
        return []

    meta_df = load_metadata(metadata_file)
    out_dir = Path(output_folder)
    out_dir.mkdir(exist_ok=True)

    results = []
    for f in files:
        # One bad file must not end the batch, whether it fails in the
        # parser, in the drawing or on the way to disk.
        try:
            data, phase_cols, error = read_gsas2_csv(f)
            if error:
                raise ValueError(error)

            name, pct, colors, window = sample_info(meta_df, f.name, f.stem)
            report_colors(phase_cols, colors)
            theta, obs, calc, bkg, resid, phases = prepare_data(
                data, phase_cols, use_sqrt=use_sqrt)
            fig = create_plot(theta, obs, calc, bkg, resid, phases, name, pct,
                              use_sqrt=use_sqrt, xlim=window, colors=colors)

            # The residual suffix is added only when it is not the default,
            # so figures made before the setting existed keep their names.
            base = (f"{f.stem}_XRD_analysis{'_sqrt' if use_sqrt else '_linear'}"
                    f"{'' if WEIGHTED_RESIDUALS else '_counts'}")
            fig.savefig(out_dir / f"{base}.pdf", bbox_inches="tight",
                        facecolor="white")
            fig.savefig(out_dir / f"{base}.png", dpi=DPI_EXPORT,
                        bbox_inches="tight", facecolor="white")
            print(f"  OK   {f.name} -> {base}.pdf / .png")
            results.append((f.name, "ok", base))

            if show:
                plt.show()
            plt.close(fig)  # release memory between files
        except Exception as e:
            print(f"  SKIP {f.name}: {e}")
            results.append((f.name, "error", str(e)))
            plt.close("all")
    print(f"Done: {sum(1 for r in results if r[1] == 'ok')}/{len(files)} "
          f"file(s) plotted, output in '{out_dir}'.")
    return results

## 2. Validation (self-test on synthetic data)

Rebuilds synthetic GSAS-II-style exports and asserts bit-exact parsing in
both separator and decimal-mark variants, detection of the phase columns
among a full set of export columns, isolation of corrupt, ragged and
incomplete files, the order and colours of the phases, the metadata
binding into the legend, the 2theta window, the unweighted residual panel
and the function behind the interactive section. Raises `AssertionError`
on any failure, so the notebook doubles as an automated test
(`jupyter nbconvert --execute`). Only synthetic data is used.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    rng = np.random.default_rng(0)

    n = 500
    x = np.linspace(10.0, 90.0, n)
    calc = 1000.0 * np.exp(-((x - 30.0) ** 2) / 2.0) + 50.0
    obs = calc + rng.normal(0.0, 5.0, n)
    bkg = np.full(n, 50.0)
    ds = (obs - calc) / 5.0

    df = pd.DataFrame({"2theta": x, "Obs": obs, "Calc": calc, "Bkg": bkg,
                       "diff/sigma": ds})
    df["Phase 1 hkl"] = pd.Series([30.0, 35.0, 50.0, 60.0])
    df["Phase 2 hkl"] = pd.Series([32.0, 38.0, 52.0])

    # Variant A: semicolon separator + decimal commas (locale export).
    f_a = tmp / "sample_A.csv"
    df.to_csv(f_a, sep=";", index=False, decimal=",")
    # Variant B: comma separator, decimal points, 'x, deg' header.
    f_b = tmp / "sample_B.csv"
    df.rename(columns={"2theta": "x, deg"}).to_csv(f_b, index=False)

    for f in (f_a, f_b):
        data, phase_cols, error = read_gsas2_csv(f)
        assert error is None, f"{f.name}: {error}"
        # 1. bit-exact round trip of the parsed doubles
        assert np.array_equal(data["x"], x), "2theta round trip not exact"
        assert np.array_equal(data["obs"], obs), "Obs round trip not exact"
        assert np.array_equal(data["resid"], ds), "diff/sigma not exact"
        # 2. structure detection
        assert sorted(phase_cols) == ["Phase 1 hkl", "Phase 2 hkl"]
        print(f"{f.name}: bit-exact parse, 2 phases - OK")

    # Degraded inputs must be isolated, not fatal.
    (tmp / "corrupt.csv").write_bytes(b"\x00\x01\x02 not a csv \xff")
    _, _, err = read_gsas2_csv(tmp / "corrupt.csv")
    assert err is not None
    print(f"corrupt.csv isolated: {err.splitlines()[0][:60]}... - OK")

    (tmp / "no_theta.csv").write_text("A;B\n1;2\n")
    _, _, err = read_gsas2_csv(tmp / "no_theta.csv")
    assert err == "2theta column not found"
    print("no_theta.csv isolated: 2theta column not found - OK")

    f_min = tmp / "only_obs.csv"
    df[["2theta", "Obs"]].to_csv(f_min, sep=";", index=False)
    _, _, err = read_gsas2_csv(f_min)
    assert err == "residual column 'diff/sigma' not found", err
    print("only_obs.csv isolated: no residual column - OK")

    f_part = tmp / "no_calc.csv"
    df[["2theta", "Obs", "diff/sigma"]].to_csv(f_part, sep=";", index=False)
    data_p, _, err = read_gsas2_csv(f_part)
    assert err is None and np.all(data_p["calc"] == 0.0)
    print("no_calc.csv: calc and bkg padded with zeros, still drawn - OK")

    f_nan = tmp / "obs_all_text.csv"
    df.assign(Obs="n/a").to_csv(f_nan, sep=";", index=False)
    _, _, err = read_gsas2_csv(f_nan)
    assert err == "no valid data in the obs column", err
    print("obs_all_text.csv isolated: nothing left to draw - OK")

    # 3. a file the C parser cannot tokenise: one row carries two fields
    # too many. The fallback drops that row and keeps the rest exact.
    lines = f_a.read_text().splitlines()
    lines[5] += ";99;99"
    f_ragged = tmp / "ragged.csv"
    f_ragged.write_text("\n".join(lines) + "\n")
    data_r, phase_r, err = read_gsas2_csv(f_ragged)
    assert err is None, f"ragged.csv: {err}"
    assert np.array_equal(data_r["obs"], np.delete(obs, 4)), "fallback not exact"
    assert sorted(phase_r) == ["Phase 1 hkl", "Phase 2 hkl"], phase_r
    print("ragged.csv: one row skipped, the rest bit-exact - OK")

    # 4. a complete export, with the header the README documents and in the
    # column order GSAS-II writes: the sparse bookkeeping columns
    # (tick-pos, Axis-limits) must not be mistaken for phases, a repeated
    # header must not either, and a column too full to be a reflection list
    # is reported rather than dropped in silence.
    full = pd.DataFrame({"used": np.ones(n), "x, 2theta (deg)": x, "obs": obs,
                         "calc": calc, "bkg": bkg, "diff": obs - calc})
    full["Alpha hkl"] = pd.Series([30.0, 35.0, 50.0, 60.0])
    full["Beta hkl"] = pd.Series([32.0, 38.0, 52.0])
    full["tick-pos"] = pd.Series([-0.5])
    full["diff/sigma"] = ds
    full["Axis-limits"] = pd.Series([10.0, 90.0])
    f_full = tmp / "full_export.csv"
    full.to_csv(f_full, sep=";", index=False)
    data_full, phase_cols_full, err = read_gsas2_csv(f_full)
    assert err is None, f"full_export.csv: {err}"
    assert sorted(phase_cols_full) == ["Alpha hkl", "Beta hkl"], phase_cols_full
    print(f"full_export.csv: {len(full.columns)} columns, 2 phases - OK")

    f_dup = tmp / "repeated_header.csv"
    f_dup.write_text("2theta;Obs;diff/sigma;tick-pos;tick-pos\n"
                     + "".join(f"{v};1;0;;\n" for v in (10.0, 20.0, 30.0))
                     .replace("10.0;1;0;;", "10.0;1;0;-0.5;-0.5"))
    _, phase_dup, err = read_gsas2_csv(f_dup)
    assert err is None and phase_dup == [], phase_dup
    print("repeated_header.csv: 'tick-pos.1' is not a phase - OK")

    f_dense = tmp / "dense_column.csv"
    dense = df.copy()
    dense["Wide hkl"] = pd.Series(np.linspace(10.0, 90.0, n - 100))
    dense.to_csv(f_dense, sep=";", index=False)
    _, phase_dense, err = read_gsas2_csv(f_dense)
    assert err is None and "Wide hkl" not in phase_dense, phase_dense
    print("dense_column.csv: too full for a reflection list, reported - OK")

    # 5. metadata wiring into the legend (synthetic metadata only).
    meta = tmp / "Samples_metadata.csv"
    meta.write_text("filename;formula;phase_1_pct;phase_2_pct\n"
                    "sample_A.csv;Sample A (synthetic);60;40\n")
    meta_df = load_metadata(meta)
    name, pct, colors, window = sample_info(meta_df, "sample_A.csv", "sample_A")
    assert name == "Sample A (synthetic)"
    assert pct == {"phase 1": 60.0, "phase 2": 40.0}, pct
    assert colors == {} and window == (None, None)

    data, phase_cols, _ = read_gsas2_csv(f_a)
    theta, o, c, b, r, phases = prepare_data(data, phase_cols, use_sqrt=True)
    fig = create_plot(theta, o, c, b, r, phases, name, pct, use_sqrt=True)
    labels = [t.get_text() for t in fig.axes[0].get_legend().get_texts()]
    assert "Phase 1 (60%)" in labels and "Phase 2 (40%)" in labels
    assert name in labels
    print("legend labels:", labels)
    plt.show()
    plt.close(fig)

    # 6. phase names, order and colours: alphabetical by legend name, one
    # tick row per phase, the cycle handed out in that order. Two columns
    # of one phase share an entry, a colour and a row.
    assert phase_label("Phase 1 hkl") == "Phase 1"
    assert phase_label("alpha") == "Alpha"
    phases3 = dict(phases, **{"Phase 3 hkl": np.array([40.0, 45.0])})
    fig3 = create_plot(theta, o, c, b, r, phases3, name, pct, use_sqrt=True)
    ax3 = fig3.axes[0]
    assert [t.get_text() for t in ax3.get_legend().get_texts()][3:] == [
        "Phase 1 (60%)", "Phase 2 (40%)", "Phase 3"]
    assert [h.get_color() for h in ax3.get_legend().legend_handles[3:]] == list(
        PHASE_COLOR_CYCLE[:3])
    assert len([c for c in ax3.collections if hasattr(c, "get_segments")]) == 3
    plt.close(fig3)

    twin = dict(phases, **{"Phase 1": phases["Phase 1 hkl"]})
    fig_t = create_plot(theta, o, c, b, r, twin, name, pct, use_sqrt=True)
    ax_t = fig_t.axes[0]
    assert [t.get_text() for t in ax_t.get_legend().get_texts()][3:] == [
        "Phase 1 (60%)", "Phase 2 (40%)"]
    assert [h.get_color() for h in ax_t.get_legend().legend_handles[3:]] == list(
        PHASE_COLOR_CYCLE[:2]), "a repeated label must not eat a colour"
    print("phase order, colour cycle and repeated labels - OK")
    plt.close(fig_t)

    # 7. a colour from the metadata follows its phase, whether or not the
    # other phases are in that sample, and the longest matching key wins.
    meta_col = tmp / "metadata_colour.csv"
    meta_col.write_text("filename;phase_2_color;phase_1_color\n"
                        "sample_A.csv;#123456;not a colour\n")
    _, _, colors_c, _ = sample_info(load_metadata(meta_col), "sample_A.csv",
                                    "sample_A")
    assert colors_c == {"phase 2": "#123456"}, colors_c  # bad cell dropped
    solo = {"Phase 2 hkl": phases["Phase 2 hkl"]}
    fig1 = create_plot(theta, o, c, b, r, solo, name, pct, use_sqrt=True,
                       colors=colors_c)
    assert fig1.axes[0].get_legend().legend_handles[3].get_color() == "#123456"
    fig2 = create_plot(theta, o, c, b, r, phases, name, pct, use_sqrt=True,
                       colors=colors_c)
    assert [h.get_color() for h in fig2.axes[0].get_legend().legend_handles[3:]
            ] == [PHASE_COLOR_CYCLE[0], "#123456"]
    assert longest_match({"phase": "#000000", "phase 1": "#ffffff"},
                         "Phase 1") == "#ffffff"
    assert metadata_keys(["_pct", "phase_1_pct"], "_pct") == {
        "phase 1": "phase_1_pct"}, "a suffix-only column matches every phase"
    print("metadata colour: pinned per phase, longest key wins - OK")
    plt.close(fig1)
    plt.close(fig2)

    # 8. the two dictionaries in the routines cell: a legend name, and a
    # colour keyed by the name that is printed.
    PHASE_LABELS = {"phase 1": "Alpha"}
    PHASE_COLORS = {"alpha": "#654321"}
    assert phase_label("Phase 1 hkl") == "Alpha"
    fig_d = create_plot(theta, o, c, b, r, phases, name, pct, use_sqrt=True)
    handles_d = fig_d.axes[0].get_legend().legend_handles[3:]
    texts_d = [t.get_text() for t in fig_d.axes[0].get_legend().get_texts()][3:]
    assert texts_d == ["Alpha", "Phase 2 (40%)"], texts_d
    assert [h.get_color() for h in handles_d] == ["#654321",
                                                  PHASE_COLOR_CYCLE[0]]
    PHASE_LABELS, PHASE_COLORS = {}, {}
    print("PHASE_LABELS and PHASE_COLORS: renamed phase keeps its colour - OK")
    plt.close(fig_d)

    # 9. metadata quirks: a decimal comma, an empty formula cell that must
    # not reach the legend as 'nan', a percentage column matching no phase,
    # and two columns matching one phase.
    meta_v = tmp / "metadata_variants.csv"
    meta_v.write_text("filename;formula;phase_1_pct;phase_9_pct\n"
                      "sample_A.csv;;60,5;10\n")
    name_v, pct_v, _, _ = sample_info(load_metadata(meta_v), "sample_A.csv",
                                      "sample_A")
    assert name_v == "sample_A", name_v
    assert pct_v == {"phase 1": 60.5, "phase 9": 10.0}, pct_v
    fig_v = create_plot(theta, o, c, b, r, phases, name_v, pct_v, use_sqrt=True)
    lab_v = [t.get_text() for t in fig_v.axes[0].get_legend().get_texts()]
    assert lab_v[0] == "sample_A" and "Phase 1 (60.5%)" in lab_v, lab_v
    assert fig_v.axes[1].get_ylabel() == r"diff/$\sigma$"
    assert phase_fraction({"phase": 40.0, "phase 1": 45.0}, "Phase 1") == 0.0
    print("metadata variants: comma, empty formula, orphan, collision - OK")
    plt.close(fig_v)

    # 10. the 2theta window: the measured range by default, the constants
    # over it, the metadata pair over both, and the intensity axis scaled
    # on what the window shows.
    assert plot_window(theta) == (10.0, 90.0), plot_window(theta)
    PLOT_X_MIN, PLOT_X_MAX = 13, 85
    assert plot_window(theta) == (13.0, 85.0), plot_window(theta)
    assert plot_window(theta, 20.0, 60.0) == (20.0, 60.0)
    PLOT_X_MIN, PLOT_X_MAX = None, None
    meta_w = tmp / "metadata_window.csv"
    meta_w.write_text("filename;x_min;x_max\nsample_A.csv;20;60,5\n")
    _, _, _, window_w = sample_info(load_metadata(meta_w), "sample_A.csv",
                                    "sample_A")
    assert window_w == (20.0, 60.5), window_w
    fig_w = create_plot(theta, o, c, b, r, phases, name, pct, use_sqrt=True,
                        xlim=window_w)
    assert fig_w.axes[1].get_xlim() == (20.0, 60.5)
    full_top = create_plot(theta, o, c, b, r, phases, name, pct,
                           use_sqrt=True).axes[0].get_ylim()[1]
    assert fig_w.axes[0].get_ylim()[1] < full_top, "y not rescaled to the window"
    print("plot window: data range, constants, metadata, y rescaled - OK")
    plt.close("all")

    # 11. the unweighted panel reads the raw diff, labels it in counts, and
    # refuses a file that does not carry the column it was asked for.
    data_u, phase_u, err = read_gsas2_csv(f_full, weighted=False)
    assert err is None, err
    assert np.array_equal(data_u["resid"], obs - calc), "diff not exact"
    fig_u = create_plot(*prepare_data(data_u, phase_u, use_sqrt=True),
                        name, pct, use_sqrt=True, weighted=False)
    assert fig_u.axes[1].get_ylabel() == r"diff / (a.u.)"
    plt.close(fig_u)
    _, _, err = read_gsas2_csv(f_a, weighted=False)
    assert err == "residual column 'diff' not found", err
    assert WEIGHTED_RESIDUALS is True, "the constant must not be mutated"
    print("unweighted residuals: raw diff, own label, missing column - OK")

    # 12. the function behind the interactive panel: limits applied, the
    # metadata line handed back, the setting left alone.
    fig_i, line_i = replot_file(f_a, meta, x_min=20, x_max=60.5, y_max=40,
                                use_sqrt=True, weighted=True)
    assert fig_i.axes[1].get_xlim() == (20.0, 60.5)
    assert fig_i.axes[0].get_ylim()[1] == 40.0
    assert line_i.splitlines()[1] == "sample_A.csv;Sample A (synthetic);20;60.5"
    plt.close(fig_i)
    fig_j, _ = replot_file(f_full, meta, weighted=False)
    assert fig_j.axes[1].get_ylabel() == r"diff / (a.u.)"
    assert WEIGHTED_RESIDUALS is True, "the constant must not be mutated"
    plt.close(fig_j)
    try:
        replot_file(f_min, meta)
    except ValueError as e:
        assert "residual column" in str(e), e
    else:
        raise AssertionError("a file without residuals must raise")
    print("replot_file: limits applied, metadata line, no side effect - OK")

    # 13. one unusable file must not end the batch: the files after it are
    # still drawn, and it is reported with a reason.
    batch_in, batch_out = tmp / "batch_in", tmp / "batch_out"
    batch_in.mkdir()
    df.to_csv(batch_in / "a_good.csv", sep=";", index=False)
    df.assign(Obs="n/a").to_csv(batch_in / "b_broken.csv", sep=";", index=False)
    df.to_csv(batch_in / "c_good.csv", sep=";", index=False)
    outcome = process_folder(batch_in, meta, batch_out, show=False)
    assert [status for _, status, _ in outcome] == ["ok", "error", "ok"], outcome
    assert len(list(batch_out.glob("*.png"))) == 2
    print("batch: the broken file is reported, the others are drawn - OK")

print("\nALL VALIDATION CHECKS PASSED")

## 3. Plot your own exports

Copy your CSV exports into `data/`, optionally place `Samples_metadata.csv`
next to the notebook, adjust the parameters below and run. Figures are
shown inline and saved to `output/` as PDF and 600 dpi PNG. Step-by-step
instructions (local and Google Colab) are in the README.

> **Keep your data private:** `data/`, `output/` and `Samples_metadata.csv`
> are listed in `.gitignore` and must never be committed or uploaded.

In [ ]:
DATA_FOLDER = "data"                       # your GSAS-II CSV exports
METADATA_FILE = "Samples_metadata.csv"     # optional, PRIVATE - never commit
OUTPUT_FOLDER = "output"                   # created automatically
USE_SQRT = True                            # False -> linear intensity axis

results = process_folder(DATA_FOLDER, METADATA_FILE, OUTPUT_FOLDER,
                         use_sqrt=USE_SQRT)

## 4. Try a different window on one file

Pick a file, type the limits, press **Apply**. Nothing is saved: this is
the place to find the window you want before running the batch again.
An empty box leaves that end of the axis to the setting behind it, the
`PLOT_X_MIN` and `PLOT_X_MAX` constants for 2theta and the data itself
for the intensity.

The panel prints a metadata line under the figure. Paste it into
`Samples_metadata.csv` and section 3 will draw that sample this way every
time.

This section needs `ipywidgets`, which the first cell installs along with
the other dependencies. Without it the section prints how to install it
and the rest of the notebook is unaffected.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except ImportError:
    widgets = None
    print("ipywidgets is not installed: run 'pip install ipywidgets', "
          "then re-run this cell.")

files = sorted(f for f in Path(DATA_FOLDER).glob("*.csv")
               if f.name != Path(METADATA_FILE).name)

if widgets is None or not files:
    if widgets is not None:
        print(f"No CSV files in '{DATA_FOLDER}': nothing to replot.")
else:
    picker = widgets.Dropdown(options=[(f.name, str(f)) for f in files],
                              description="File:")
    boxes = {k: widgets.Text(description=d, placeholder="auto",
                             layout=widgets.Layout(width="180px"))
             for k, d in (("x_min", "2theta min"), ("x_max", "2theta max"),
                          ("y_min", "y min"), ("y_max", "y max"))}
    sqrt_box = widgets.Checkbox(value=USE_SQRT, description="sqrt intensity")
    weighted_box = widgets.Checkbox(value=WEIGHTED_RESIDUALS,
                                    description="diff/sigma")
    apply_button = widgets.Button(description="Apply", button_style="primary")
    out = widgets.Output()

    def on_apply(_):
        """Redraw the picked file with whatever the boxes currently hold."""
        with out:
            clear_output(wait=True)
            limits = {k: to_number(b.value) if b.value.strip() else None
                      for k, b in boxes.items()}
            try:
                fig, line = replot_file(picker.value, METADATA_FILE,
                                        use_sqrt=sqrt_box.value,
                                        weighted=weighted_box.value, **limits)
            except ValueError as e:
                print(f"Cannot draw this file: {e}")
                return
            plt.show()
            plt.close(fig)
            print("Metadata line for this window:\n" + line)

    apply_button.on_click(on_apply)
    display(widgets.VBox([
        picker,
        widgets.HBox([boxes["x_min"], boxes["x_max"]]),
        widgets.HBox([boxes["y_min"], boxes["y_max"]]),
        widgets.HBox([sqrt_box, weighted_box, apply_button]),
        out,
    ]))